In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Mount Google Drive & Install Libs
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers==4.47.0",   # ← Fixed: supports EncoderDecoderCache
    "datasets==3.2.0",
    "evaluate==0.4.3",
    "scikit-learn",
    "torch",
    "accelerate==1.2.1",
    "peft"
], check=True)

# ⚠️ IMPORTANT: After install, go to Runtime → Restart Session
# Then run ALL cells again from Cell 1
print("✅ Libraries installed — now restart the runtime!")

Mounted at /content/drive
✅ Libraries installed — now restart the runtime!


In [2]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
import torch
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from datasets import Dataset

# Suppress noisy warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ── Paths ──────────────────────────────────────────────────
DATA_DIR    = "/content/drive/MyDrive/Data/LiarData"
OUTPUT_DIR  = "/content/drive/MyDrive/Data/LiarData/finetuned_model"
LOG_DIR     = "/content/drive/MyDrive/Data/LiarData/logs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,    exist_ok=True)

# ── Model ──────────────────────────────────────────────────
BASE_MODEL  = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"

# ── Training Hyper-parameters ──────────────────────────────
MAX_LEN     = 256   # Enough for claim + justification; longer = slower
BATCH_SIZE  = 8     # Safe for a T4 GPU; drop to 4 if OOM
GRAD_ACC    = 4     # Effective batch = 32
LR          = 2e-5
EPOCHS      = 5
WARMUP      = 0.1   # 10 % of steps used for warm-up

print("✅ Config ready.")
print(f"   Output → {OUTPUT_DIR}")

✅ Config ready.
   Output → /content/drive/MyDrive/Data/LiarData/finetuned_model


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — LIAR-Plus Column Schema & Label Map
# ─────────────────────────────────────────────
# LIAR-Plus TSV has 15 columns (no header row):
COLUMNS = [
    "id",               # 0
    "label",            # 1  ← gold label (6 classes)
    "statement",        # 2  ← the CLAIM  (hypothesis)
    "subjects",         # 3
    "speaker",          # 4
    "speaker_job",      # 5
    "state_info",       # 6
    "party",            # 7
    "barely_true_cnt",  # 8
    "false_cnt",        # 9
    "half_true_cnt",    # 10
    "mostly_true_cnt",  # 11
    "pants_fire_cnt",   # 12
    "context",          # 13
    "justification",    # 14  ← the EVIDENCE (premise) — LIAR-Plus addition
]

# Map 6 LIAR labels → 3 NLI classes used by the base model
#   entailment   (0) → claim is supported by evidence
#   neutral      (1) → claim is neither confirmed nor denied
#   contradiction(2) → claim is refuted by evidence
LABEL_MAP = {
    "true":        0,   # entailment
    "mostly-true": 0,   # entailment
    "half-true":   1,   # neutral
    "barely-true": 2,   # contradiction
    "false":       2,   # contradiction
    "pants-fire":  2,   # contradiction  (pants-on-fire)
}

ID2LABEL = {0: "Entailment (Verified)",
            1: "Neutral (Inconclusive)",
            2: "Contradiction (Fake)"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print("✅ Label mapping defined.")

✅ Label mapping defined.


In [4]:
# ─────────────────────────────────────────────
# CELL 4 — Data Loading & Preprocessing
# ─────────────────────────────────────────────
def load_liar_plus(split_name: str) -> pd.DataFrame:
    """
    Loads a LIAR-Plus TSV file and returns a clean DataFrame.
    split_name: 'train2' | 'test2' | 'val2'
    """
    path = os.path.join(DATA_DIR, f"{split_name}.tsv")
    print(f"   Loading {path} …", end=" ")

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=COLUMNS,
        dtype=str,          # Read everything as string first; avoids type errors
        on_bad_lines="skip",
    )

    # ── 1. Drop rows with missing critical fields ──────────
    before = len(df)
    df = df.dropna(subset=["label", "statement"])
    dropped = before - len(df)
    if dropped:
        print(f"\n   ⚠  Dropped {dropped} rows with missing label/statement.")

    # ── 2. Normalise label strings ─────────────────────────
    df["label"] = df["label"].str.strip().str.lower()

    # ── 3. Keep only rows whose label we know how to map ──
    df = df[df["label"].isin(LABEL_MAP)]

    # ── 4. Integer NLI label ───────────────────────────────
    df["nli_label"] = df["label"].map(LABEL_MAP)

    # ── 5. Clean text fields ───────────────────────────────
    def clean(text):
        if pd.isna(text) or str(text).strip().lower() in ("none", "nan", "", "n/a"):
            return ""
        return str(text).strip()

    df["statement"]     = df["statement"].apply(clean)
    df["justification"] = df["justification"].apply(clean)
    df["context"]       = df["context"].apply(clean)
    df["speaker"]       = df["speaker"].apply(clean)

    # ── 6. Build the PREMISE & HYPOTHESIS ─────────────────
    def build_premise(row):
        if row["justification"]:
            return row["justification"]
        if row["context"]:
            return row["context"]
        return "DROP" # Mark for dropping

    df["premise"] = df.apply(build_premise, axis=1)
    df["hypothesis"] = df["statement"]   # ← Added this crucial line back in!

    # ── 7. Remove rows where premise is empty ─────────────
    df = df[df["premise"] != "DROP"] # Strictly remove rows without text evidence

    print(f"✅  {len(df)} usable rows.")
    return df[["premise", "hypothesis", "nli_label", "label"]].reset_index(drop=True)


print("Loading datasets …")
df_train = load_liar_plus("train2")
df_val   = load_liar_plus("val2")
df_test  = load_liar_plus("test2")

print(f"\n📊 Split sizes:")
print(f"   Train : {len(df_train):>5}")
print(f"   Val   : {len(df_val):>5}")
print(f"   Test  : {len(df_test):>5}")

print("\n📊 Train label distribution:")
print(df_train["label"].value_counts())

Loading datasets …
   Loading /content/drive/MyDrive/Data/LiarData/train2.tsv … 
   ⚠  Dropped 2 rows with missing label/statement.
✅  10237 usable rows.
   Loading /content/drive/MyDrive/Data/LiarData/val2.tsv … ✅  1284 usable rows.
   Loading /content/drive/MyDrive/Data/LiarData/test2.tsv … ✅  1266 usable rows.

📊 Split sizes:
   Train : 10237
   Val   :  1284
   Test  :  1266

📊 Train label distribution:
label
half-true      2114
false          1993
mostly-true    1962
true           1676
barely-true    1654
pants-fire      838
Name: count, dtype: int64


In [5]:
# ─────────────────────────────────────────────
# CELL 5 — Tokenisation
# ─────────────────────────────────────────────
print(f"\nLoading tokenizer from: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    """
    NLI format: tokenizer(premise, hypothesis)
    The model was trained with premise as sequence A
    and hypothesis as sequence B — keep that order.
    """
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,   # Dynamic padding via DataCollatorWithPadding
    )

def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        df[["premise", "hypothesis", "nli_label"]].rename(
            columns={"nli_label": "labels"}
        )
    )
    ds = ds.map(tokenize, batched=True, remove_columns=["premise", "hypothesis"])
    return ds

print("Tokenising splits …")
train_ds = df_to_hf_dataset(df_train)
val_ds   = df_to_hf_dataset(df_val)
test_ds  = df_to_hf_dataset(df_test)

print(f"✅ Tokenisation complete.")
print(f"   Sample features: {list(train_ds.features.keys())}")


Loading tokenizer from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli


tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

...

Tokenising splits …


Map:   0%|          | 0/10237 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1266 [00:00<?, ? examples/s]

✅ Tokenisation complete.
   Sample features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [6]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch
import gc

if 'model' in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Load the base model normally
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
)
# Safely update the label names AFTER the pre-trained weights are secured
base_model.config.id2label = ID2LABEL
base_model.config.label2id = LABEL2ID

# Define the LoRA Configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,               # Rank of the adapter matrices
    lora_alpha=64,      # Scaling factor
    lora_dropout=0.1,   # Dropout for regularization
    bias="none",
)

# Wrap the base model with the LoRA adapters
model = get_peft_model(base_model, lora_config)

# This will print out exactly how few parameters you are training
model.print_trainable_parameters()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ LoRA Model loaded → {device}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

trainable params: 3,148,803 || all params: 438,213,638 || trainable%: 0.7186
✅ LoRA Model loaded → cuda


In [7]:
# ─────────────────────────────────────────────
# CELL 7 — Metrics
# ─────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="weighted"
    )["f1"]

    return {"accuracy": acc, "f1_weighted": f1}

In [8]:
# ─────────────────────────────────────────────
# CELL 8 — Training Arguments (IMPROVED)
# ─────────────────────────────────────────────
import torch
from torch import nn
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from transformers import DataCollatorWithPadding

# ── Fix #2: Compute class weights to handle imbalance ──
from sklearn.utils.class_weight import compute_class_weight

train_labels = df_train["nli_label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class weights: {class_weights.round(3)}")
# Neutral will get a higher weight since it has fewer samples


# ── Custom Trainer with weighted loss ──────────────────
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
        # Ensure labels are explicitly long/integers
        loss = loss_fn(logits, labels.long())

        return (loss, outputs) if return_outputs else loss


# ── Fix #1 & #3: LoRA LR, Cosine Scheduler + gradient clipping ──────────
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 4,           # ← Set exactly to 4 epochs
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACC,

    learning_rate               = 2e-4,        # ← Crucial for LoRA
    lr_scheduler_type           = "cosine",    # ← Drops the LR smoothly to land perfectly
    warmup_ratio                = 0.15,
    weight_decay                = 0.05,
    max_grad_norm               = 1.0,

    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",     # ← Checkpoints stay ON
    load_best_model_at_end      = True,        # ← Automatically grabs your best epoch!
    metric_for_best_model       = "f1_weighted",
    greater_is_better           = True,
    save_total_limit            = 2,           # ← Deletes older checkpoints to save Drive space

    logging_dir                 = LOG_DIR,
    logging_steps               = 50,
    report_to                   = "none",

    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    seed                        = 42,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(                     # ← Use weighted trainer
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("✅ Improved trainer configured.")

Class weights: [0.938 1.614 0.761]
✅ Improved trainer configured.


In [9]:
print("\n🚀 Starting fine-tuning …\n")
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"   Total steps       : {train_result.global_step}")
print(f"   Training loss     : {train_result.training_loss:.4f}")

# Save the best model & tokenizer to Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n💾 Best model saved → {OUTPUT_DIR}")


🚀 Starting fine-tuning …

{'loss': 2.8382, 'grad_norm': 20.747333526611328, 'learning_rate': 5e-05, 'epoch': 0.15625}
{'loss': 1.3048, 'grad_norm': 8.370560646057129, 'learning_rate': 0.00010208333333333333, 'epoch': 0.3125}
{'loss': 1.1031, 'grad_norm': 10.789628982543945, 'learning_rate': 0.00015416666666666668, 'epoch': 0.46875}
{'loss': 1.1019, 'grad_norm': 4.535674571990967, 'learning_rate': 0.00019998957815945962, 'epoch': 0.625}
{'loss': 1.0975, 'grad_norm': 7.5434393882751465, 'learning_rate': 0.0001987415836360051, 'epoch': 0.78125}
{'loss': 1.0749, 'grad_norm': 8.711194038391113, 'learning_rate': 0.0001954389878558401, 'epoch': 0.9375}
{'eval_loss': 0.9956945776939392, 'eval_accuracy': 0.530373831775701, 'eval_f1_weighted': 0.5028059644019285, 'eval_runtime': 23.2086, 'eval_samples_per_second': 55.324, 'eval_steps_per_second': 3.49, 'epoch': 1.0}
{'loss': 1.0609, 'grad_norm': 14.406938552856445, 'learning_rate': 0.0001901505107765596, 'epoch': 1.09375}
{'loss': 1.0512, 'grad

In [10]:
print("\n📊 Evaluating on held-out test set …")
test_results = trainer.predict(test_ds)

preds  = np.argmax(test_results.predictions, axis=-1)
labels = test_results.label_ids

# Overall metrics
test_acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
test_f1  = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]

print(f"\n   Test Accuracy    : {test_acc:.4f}")
print(f"   Test F1 (weighted): {test_f1:.4f}")

# Per-class report
target_names = [ID2LABEL[i] for i in range(3)]
print("\n" + classification_report(labels, preds, target_names=target_names))

# Save metrics to a text file on Drive
metrics_path = os.path.join(OUTPUT_DIR, "test_metrics.txt")
with open(metrics_path, "w") as f:
    f.write(f"Test Accuracy     : {test_acc:.4f}\n")
    f.write(f"Test F1 (weighted): {test_f1:.4f}\n\n")
    f.write(classification_report(labels, preds, target_names=target_names))
print(f"📄 Metrics saved → {metrics_path}")


📊 Evaluating on held-out test set …

   Test Accuracy    : 0.5774
   Test F1 (weighted): 0.5842

                        precision    recall  f1-score   support

 Entailment (Verified)       0.61      0.68      0.64       449
Neutral (Inconclusive)       0.30      0.36      0.33       264
  Contradiction (Fake)       0.74      0.59      0.66       553

              accuracy                           0.58      1266
             macro avg       0.55      0.55      0.54      1266
          weighted avg       0.60      0.58      0.58      1266

📄 Metrics saved → /content/drive/MyDrive/Data/LiarData/finetuned_model/test_metrics.txt


In [ ]:
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True",      fontsize=12)
ax.set_title("Confusion Matrix — Test Set", fontsize=14)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
fig.savefig(cm_path, dpi=150)
plt.show()
print(f"🖼  Confusion matrix saved → {cm_path}")

In [2]:
# --- THE ABSOLUTE SILENCE BLOCK ---
import os
import warnings
import logging

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

def block_all_warnings(message, category, filename, lineno, file=None, line=None):
    pass
warnings.showwarning = block_all_warnings

logging.getLogger().setLevel(logging.ERROR)
logging.getLogger("jupyter_client").setLevel(logging.ERROR)
# ----------------------------------

# 1. Install required libraries
!pip install -q transformers torch gradio requests peft

import gradio as gr
import torch
import requests
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# 2. Load the Base Model + Your Fine-Tuned LoRA Adapter
base_model_name = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
adapter_path = "/content/drive/MyDrive/Data/LiarData/finetuned_model"

print(f"Loading Base Model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=3
)

print(f"Applying Your LoRA Weights...")
# This stitches your custom training on top of the base model
model = PeftModel.from_pretrained(base_model, adapter_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# Set to evaluation mode
model.eval()
print("✅ Application AI loaded successfully!")

# 3. Define the Web Search Function using Google Serper
def get_serper_evidence(claim, api_key, max_results=6):
    """Searches Google via Serper.dev and concatenates snippets."""
    if not api_key or not api_key.strip():
        return "Error: Please enter your Serper API Key."

    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": claim})
    headers = {
        'X-API-KEY': api_key.strip(),
        'Content-Type': 'application/json'
    }

    try:
        response = requests.post(url, headers=headers, data=payload)

        if response.status_code == 403:
            return "Error: Invalid Serper API Key."
        response.raise_for_status()

        results = response.json()

        if "organic" not in results or len(results["organic"]) == 0:
            return "No internet evidence found on Google."

        evidence_snippets = []
        seen_texts = set()

        for i, res in enumerate(results["organic"][:max_results]):
            if "snippet" in res:
                snippet_text = res['snippet'].strip()
                if snippet_text not in seen_texts:
                    seen_texts.add(snippet_text)
                    evidence_snippets.append(f"Source {len(evidence_snippets)+1}: {snippet_text}")

        if not evidence_snippets:
            return "No readable text snippets found in the search results."

        return " ".join(evidence_snippets)

    except Exception as e:
        return f"Error fetching search results: {str(e)}"

# 4. Define the Core Prediction Logic
def analyze_fake_news(claim, api_key):
    if not claim.strip():
        return "Please enter a valid claim.", {}, ""

    # Step A: Fetch Live Evidence from Serper
    evidence = get_serper_evidence(claim, api_key)

    if "Error" in evidence or "No internet evidence" in evidence:
        return "SEARCH FAILED", {"Error": 1.0}, evidence

    # Step B: Prepare inputs for the model
    inputs = tokenizer(evidence, claim, truncation=True, max_length=512, return_tensors="pt").to(device)

    # Step C: Model Inference
    with torch.no_grad():
        output = model(**inputs)

    probabilities = torch.softmax(output.logits, dim=-1).tolist()[0]

    results_dict = {
        "Verified (Evidence Supports Claim)": float(probabilities[0]),
        "Neutral (Inconclusive)": float(probabilities[1]),
        "Fake News (Evidence Refutes Claim)": float(probabilities[2])
    }

    # Step D: Determine the final verdict string
    verdict_index = probabilities.index(max(probabilities))
    if verdict_index == 0:
        final_verdict = "✅ VERIFIED: Google Search evidence supports this claim."
    elif verdict_index == 2:
        final_verdict = "❌ FAKE NEWS: Google Search evidence completely refutes this claim."
    else:
        final_verdict = "⚠️ INCONCLUSIVE: Google search results do not definitively prove or disprove this."

    return final_verdict, results_dict, evidence

# 5. Build the Gradio Application UI
with gr.Blocks() as demo:
    gr.Markdown("# 🔍 AI Fake News & Fact-Checking Application")
    gr.Markdown("Enter a claim below. The system will search Google for live evidence and use a Deep Learning model to determine if the claim is True or Fake.")
    gr.Markdown("This application leverages an advanced DeBERTa model combined with LoRA fine-tuning to compare your claim against live internet search results and detect misinformation in real-time.")

    with gr.Row():
        with gr.Column(scale=2):
            api_input = gr.Textbox(
                label="Serper.dev API Key",
                placeholder="Paste your API key here...",
                type="password"
            )
            claim_input = gr.Textbox(
                label="Enter the Claim to Fact-Check",
                placeholder="e.g., The Eiffel Tower was originally intended for Barcelona.",
                lines=3
            )
            submit_btn = gr.Button("Analyze Claim", variant="primary")

        with gr.Column(scale=3):
            verdict_output = gr.Textbox(label="Final Verdict")
            confidence_output = gr.Label(label="AI Confidence Scores", num_top_classes=3)

    with gr.Row():
        evidence_output = gr.Textbox(label="Evidence Retrieved from Google Serper", lines=6, interactive=False)

    submit_btn.click(
        fn=analyze_fake_news,
        inputs=[claim_input, api_input],
        outputs=[verdict_output, confidence_output, evidence_output]
    )

# 6. Launch the App
demo.launch(debug=True, share=True)

Loading Base Model...


tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Applying Your LoRA Weights...
✅ Application AI loaded successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://209e2ea757798f579d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://209e2ea757798f579d.gradio.live
